[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/61_multi_token_prediction_solution.ipynb)

# 🟡 Solution: Multi-Token Prediction Loss

Reference solution for the Multi-Token Prediction (MTP) training objective, which trains N independent prediction heads in parallel to predict N future tokens simultaneously.

$$\mathcal{L}_{\text{MTP}} = \frac{1}{N} \sum_{i=1}^{N} \mathcal{L}_{\text{CE}}(H W_i^\top, y_{:,:,i})$$

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def multi_token_prediction_loss(hidden_states, heads, targets):
    B, S, D = hidden_states.shape
    N = len(heads)
    total_loss = 0.0
    for i, head in enumerate(heads):
        logits = hidden_states @ head
        logits_max = logits.max(dim=-1, keepdim=True).values
        shifted = logits - logits_max
        log_probs = shifted - torch.log(torch.exp(shifted).sum(dim=-1, keepdim=True))
        tgt = targets[:, :, i]
        log_p = log_probs.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
        total_loss = total_loss + (-log_p.mean())
    return total_loss / N

In [ ]:
torch.manual_seed(0)
B, S, D, V = 2, 5, 16, 10

hidden = torch.randn(B, S, D)
head_single = torch.randn(D, V)
targets_single = torch.randint(0, V, (B, S, 1))

# N=1 should match standard cross-entropy
mtp_loss = multi_token_prediction_loss(hidden, [head_single], targets_single)

logits = hidden @ head_single
ce_loss = torch.nn.functional.cross_entropy(logits.reshape(-1, V), targets_single[:, :, 0].reshape(-1))

print(f"MTP loss (N=1):  {mtp_loss.item():.6f}")
print(f"Standard CE:     {ce_loss.item():.6f}")
print(f"N=1 matches CE:  {torch.allclose(mtp_loss, ce_loss, atol=1e-5)}")

# N=3 heads
heads_3 = [torch.randn(D, V) for _ in range(3)]
targets_3 = torch.randint(0, V, (B, S, 3))
loss_3 = multi_token_prediction_loss(hidden, heads_3, targets_3)
print(f"MTP loss (N=3):  {loss_3.item():.6f}")

In [ ]:
from torch_judge import check
check("multi_token_prediction")